# Online Design with Gaussian Process Tutorial

This tutorial demonstrates how to use a Gaussian Process (GP) model for **online design optimization** using the ALF (Active Learning Framework) codebase. We'll actively learn a sinusoidal function by iteratively selecting informative points to query from an online simulator.

### What is Online Design with Gaussian Processes?

Online design combines active learning with GP regression:
- **GP provides uncertainty estimates** that guide which points to query next
- **Oracle simulator** generates labels on-demand for selected points
- **Active learning** efficiently explores the function space
- **Iterative improvement** as the GP learns from newly acquired data

This is particularly powerful because:
- GPs naturally quantify uncertainty (epistemic uncertainty)
- High uncertainty regions can be preferentially explored
- We can efficiently learn functions with fewer queries than random sampling

### Tutorial Overview

In this tutorial, we'll:
1. Create an online sinusoidal simulator that acts as an oracle
2. Set up a small initial training dataset
3. Configure a GP surrogate model with uncertainty quantification
4. Create a custom search protocol for continuous spaces
5. Use an acquisition strategy to select informative points
6. Run multiple rounds of active learning (ask → oracle query → tell)
7. Visualize how the GP learns the function over time
8. Compare active learning efficiency vs random sampling

### Framework Components

Before we start, let's understand the key components:

1. **Online Simulator** (`SinusoidalSimulator`): A BaseModel that computes sin(x) with noise on-demand, simulating expensive experiments.

2. **Oracle** (`Oracle`): Wraps the simulator to evaluate candidate points during the design loop.

3. **Surrogate Model** (`GPModelTrainer`): GP that learns from observations and provides uncertainty estimates.

4. **Acquisition Function** (`Greedy`): Selects which points to query next based on GP predictions.

5. **Search Strategy** (`ContinuousGridSearch`): Custom protocol that generates candidate points in continuous space (unlike `SingleMutantSearch` which is for discrete sequences).

6. **Optimizer** (`Optimizer`): Combines acquisition and search to handle the ask-tell cycle.

7. **Task** (`DesignTask`): Orchestrates the multi-round active learning loop.

### Step 0: Environment Setup

First, let's set up the Python environment using `uv` and the project's `pyproject.toml`. This will install all required dependencies including `alf_core`, `alf_tools`, GPyTorch, and other necessary packages.

To create an environment and install the required dependencies run:

```
uv sync
```

from the `/tutorials` directory. You can also run `uv sync --extra torch_gpu` or `uv sync --extra torch_cpu` to install the dependencies for GPU or CPU respectively.

To activate your environment use `source .venv/bin/activate` from the tutorials directory. When starting the notebook, select .venv when prompted to select a kernel.

### Step 1: Import Required Libraries

Let's start by importing all the necessary components from the ALF framework:

In [ ]:
# Core framework imports
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from alf_core import (
    BaseDatasetConfig,
    DesignTask,
    FileTaskStateLogger,
    Optimizer,
    Oracle,
    ProtocolSearch,
    SearchProtocol,
    Surrogate,
    TaskState,
    TerminalTaskStateLogger,
)
from alf_core.dataclasses.candidate import Candidate, Modality
from alf_core.dataclasses.labeled_candidates import LabeledCandidates
from alf_core.dataclasses.predictions import Predictions
from alf_core.dataset.base_dataset import BaseDataset
from alf_core.model.base_model import BaseModel
from alf_tools.models import FeaturizerConfig, GPModelConfig, GPModelTrainer, GPTrainConfig

print("✅ All imports successful!")

### Step 1.5: Configuration Parameters

Before we proceed, let's define all the configuration parameters in one place. This makes it easy to experiment with different settings without modifying code throughout the notebook.

In [ ]:
# ============================================================
# CONFIGURATION PARAMETERS
# ============================================================
# Modify these parameters to experiment with different settings

# --- Random Seeds ---
RANDOM_SEED = 42  # Main random seed for reproducibility
RANDOM_COMPARISON_SEED = 999  # Seed for random sampling comparison

# --- Simulator Configuration ---
SIMULATOR_NOISE_STD = 0.15  # Standard deviation of observation noise

# --- Initial Data Configuration ---
N_INITIAL_SAMPLES = 20  # Number of initial training points

# --- Dataset Split Configuration ---
TRAIN_RATIO = 0.8  # Use all initial data for training
VALIDATION_FRAC = 0.2  # Fraction of training data for validation
TEST_RATIO = 0.2  # No test set (evaluate on dense grid instead)

# --- GP Model Configuration ---
GP_KERNEL_TYPE = "matern"  # Kernel type: "rbf", "matern", etc.
GP_ARD = False  # Automatic Relevance Determination (for multi-dim inputs)
GP_MEAN_TYPE = "constant"  # Mean function type: "constant" or "zero"

# --- GP Training Configuration ---
GP_LEARNING_RATE = 0.1  # Learning rate for GP hyperparameter optimization
GP_NUM_ITERATIONS = 100  # Number of training iterations
GP_OPTIMIZER_TYPE = "adam"  # Optimizer: "adam" or "sgd"
GP_LOG_FREQUENCY = 25  # Print training log every N iterations

# --- Search Space Configuration ---
X_MIN = 0  # Minimum x value in domain
X_MAX = 2 * np.pi  # Maximum x value in domain
CANDIDATE_GRID_SIZE = 500  # Number of candidate points in grid

# --- Acquisition Function Configuration ---
ACQ_THRESHOLD = 0.5  # Threshold for ThresholdUCB acquisition
ACQ_BETA = 0.8  # Beta parameter for UCB trade-off
ACQ_EXPLORATION_WEIGHT = 0.9  # Weight for exploration vs exploitation
ACQ_DIVERSITY_PENALTY = 2.0  # Diversity penalty to spread batch points apart (0=no penalty)
ACQ_DIVERSITY_LENGTH_SCALE = 0.5  # Length scale for diversity penalty (smaller=more localized)

# --- Active Learning Configuration ---
NUM_ACQ_ROUNDS = 10  # Number of active learning rounds
ACQ_BATCH_SIZE = 5  # Number of points to acquire per round

# --- Visualization Configuration ---
N_TEST_POINTS = 200  # Number of points for dense evaluation/plotting

print("✅ Configuration parameters loaded!")
print("📋 Key settings:")
print(f"   - Random seed: {RANDOM_SEED}")
print(f"   - Initial samples: {N_INITIAL_SAMPLES}")
print(f"   - Acquisition rounds: {NUM_ACQ_ROUNDS}")
print(f"   - Batch size: {ACQ_BATCH_SIZE}")
print(f"   - Total points to acquire: {NUM_ACQ_ROUNDS * ACQ_BATCH_SIZE}")
print(f"   - GP training iterations: {GP_NUM_ITERATIONS}")

### Step 2: Create the Online Sinusoidal Simulator

Instead of pre-generating data, we'll create an online simulator that computes sin(x) on-demand. This simulates a real experimental setup where we can only obtain labels by running expensive evaluations.

In [ ]:
from jaxtyping import Int


def sample_function(x: Int[torch.Tensor, " batch_size"]):
    y = np.sin(x) + np.sin(2 * x) + np.sin(x**2) + np.cos(2 * x**3)
    return y


class SinusoidalSimulator(BaseModel):
    """Online simulator for sinusoidal function.

    This simulator acts as an oracle that can evaluate any point in the input space.
    It computes y = sin(x) + noise, simulating an expensive evaluation.
    """

    def __init__(self, noise_std: float = 0.1, seed: int = 42):
        """Initialize the sinusoidal simulator.

        Args:
            noise_std: Standard deviation of Gaussian noise
            seed: Random seed for reproducibility
        """
        self.noise_std = noise_std
        self.seed = seed
        self.rng = np.random.RandomState(seed)
        self.num_evaluations = 0

    def featurise(self, inputs):
        """Convert inputs into feature representations (not used for simulator)."""
        pass

    def train(self, train_data: LabeledCandidates, val_data: LabeledCandidates = None) -> None:
        """Simulator doesn't need training."""
        pass

    def predict(self, candidate_points: list[Candidate]) -> Predictions:
        """Evaluate the sinusoidal function at given points.

        This simulates querying an expensive oracle/experiment.

        Args:
            candidate_points: List of Candidate objects with x values

        Returns:
            Predictions containing y = sin(x) + noise
        """
        # Extract x values from candidates
        x = np.array([candidate.data for candidate in candidate_points]).flatten()

        # Compute sinusoidal function with noise
        y = sample_function(x) + self.rng.randn(len(x)) * self.noise_std

        # Track number of evaluations
        self.num_evaluations += len(x)

        # Return predictions (means only, no variance from simulator)
        return Predictions(means=y)

    def sample(self, condition=None) -> list[Candidate]:
        """Sample candidate points from the model (not used for simulator)."""
        raise NotImplementedError("Sampling not implemented for simulator")

    def get_training_summary_metrics(self):
        """Return evaluation metrics."""
        return {"num_evaluations": self.num_evaluations}


# Create the simulator
simulator = SinusoidalSimulator(noise_std=SIMULATOR_NOISE_STD, seed=RANDOM_SEED)

print("✅ Online sinusoidal simulator created!")
print("🔬 Simulator properties:")
print(f"   - Noise std: {simulator.noise_std}")
print("   - Evaluation mode: On-demand (not pre-computed)")

### Step 3: Generate Initial Training Data

For online design, we start with a small initial training set. The active learning loop will then acquire more points strategically.

In [ ]:
def generate_initial_training_data(n_samples: int = 10, seed: int = 42):
    """Generate a small initial training set by querying the simulator.

    Args:
        n_samples: Number of initial training points
        seed: Random seed for reproducibility

    Returns:
        LabeledCandidates with initial training data
    """
    np.random.seed(seed)

    # Sample random x values from the domain [X_MIN, X_MAX]
    x_train = np.random.uniform(X_MIN, X_MAX, n_samples)

    # Create candidates
    candidates = [Candidate(data=np.array([xi]), modality=Modality.TABULAR) for xi in x_train]

    # Query the simulator to get labels
    predictions = simulator.predict(candidates)

    return LabeledCandidates(candidates, predictions.means), x_train


# Generate initial training data
initial_data, x_initial = generate_initial_training_data(
    n_samples=N_INITIAL_SAMPLES, seed=RANDOM_SEED
)

print("✅ Initial training data generated!")
print("📊 Initial dataset:")
print(f"   - Number of samples: {len(initial_data)}")
print(f"   - Input range: [{x_initial.min():.2f}, {x_initial.max():.2f}]")
print(f"   - Output range: [{initial_data.labels.min():.2f}, {initial_data.labels.max():.2f}]")
print(f"   - Simulator evaluations so far: {simulator.num_evaluations}")

### Step 4: Visualize Initial Data

Let's see what our initial sparse dataset looks like:

In [ ]:
# Create a dense grid for visualization
x_dense = np.linspace(X_MIN, X_MAX, N_TEST_POINTS)
y_true = sample_function(x_dense)

plt.figure(figsize=(10, 6))
plt.scatter(
    x_initial,
    initial_data.labels,
    alpha=0.7,
    label="Initial training data",
    s=100,
    color="red",
    zorder=3,
)
plt.plot(x_dense, y_true, "b--", label="True function: sin(x)", linewidth=2, alpha=0.7)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.title("Initial Training Data (Sparse)", fontsize=14, fontweight="bold")
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📈 Active learning will strategically select more points to query from the simulator.")

### Step 5: Create a Custom Dataset Class

We'll create a dataset class that holds our initial training data. Note that there's no candidate pool - candidates will be generated on-the-fly by the search strategy.

In [ ]:
class SinusoidalDataset(BaseDataset):
    """Dataset for online sinusoidal design."""

    def __init__(self, config: BaseDatasetConfig, initial_data: LabeledCandidates):
        """Initialize the dataset.

        Args:
            config: Configuration for the dataset
            initial_data: Initial labeled training data
        """
        self.initial_data = initial_data
        super().__init__(config)
        self.setup()

    def load_dataset(self) -> LabeledCandidates:
        """Load the initial dataset."""
        return self.initial_data


# Create dataset configuration
# For online design, we use all data for training (no test split needed initially)
dataset_config = BaseDatasetConfig(
    name="sinusoidal_online",
    modality=Modality.TABULAR,
    seed=RANDOM_SEED,
    train_ratio=TRAIN_RATIO,  # Use all initial data for training
    validation_frac=VALIDATION_FRAC,  # Fraction of training for validation
    test_ratio=TEST_RATIO,  # No test set (will evaluate on dense grid later)
    split_type="random",
)

# Initialize the dataset
dataset = SinusoidalDataset(dataset_config, initial_data)

print("✅ Dataset initialized!")
print("📊 Dataset splits:")
print(f"   - Training samples: {len(dataset.train_dataset)}")
print(f"   - Validation samples: {len(dataset.validation_dataset)}")
print(
    f"   - Candidate pool: {len(dataset.candidate_pool)} (empty - candidates generated on-the-fly)"
)

### Step 6: Configure the GP Surrogate Model

Now let's set up the Gaussian Process surrogate model that will learn from the observations:

In [ ]:
# Configure GP model architecture
model_config = GPModelConfig(
    kernel_type=GP_KERNEL_TYPE,  # Kernel type
    ard=GP_ARD,  # Automatic Relevance Determination
    mean_type=GP_MEAN_TYPE,  # Mean function
)

# Configure training
train_config = GPTrainConfig(
    learning_rate=GP_LEARNING_RATE,
    num_iterations=GP_NUM_ITERATIONS,
    optimizer_type=GP_OPTIMIZER_TYPE,
    log_frequency=GP_LOG_FREQUENCY,
)


# Configure featurization
def tabular_featurizer(candidates: list[Candidate]) -> torch.Tensor:
    """Convert candidate data to feature tensor."""
    features = torch.tensor([c.data for c in candidates], dtype=torch.float32)
    if features.dim() == 1:
        features = features.unsqueeze(-1)
    return features


featurizer_config = FeaturizerConfig(
    featurizer_type="custom",
    custom_featurizer=tabular_featurizer,
)

# Create the GP model
gp_model = GPModelTrainer(
    name="gp_sinusoidal_online",
    model_config=model_config,
    train_config=train_config,
    featurizer_config=featurizer_config,
)

# Wrap in Surrogate
surrogate = Surrogate(model=gp_model)

print("✅ GP surrogate model configured!")
print("🧠 Model configuration:")
print(f"   - Kernel: {model_config.kernel_type.upper()}")
print("   - Provides uncertainty estimates: Yes")
print(f"   - Training iterations: {train_config.num_iterations}")

### Step 7: Set Up the Oracle

The oracle wraps our online simulator and will be queried during the design loop:

In [ ]:
# Create oracle with the simulator
oracle = Oracle(scorer=simulator)

print("✅ Oracle initialized!")
print("🔬 Oracle properties:")
print("   - Type: Online simulator (not dataset lookup)")
print("   - Evaluates candidates on-demand")
print("   - Simulates expensive experimental evaluation")

### Step 8: Create a Custom Search Protocol for Continuous Spaces

Since `SingleMutantSearch` is designed for discrete sequences (proteins), we need a custom search protocol for continuous input spaces. Let's create `ContinuousGridSearch`:

In [ ]:
class ContinuousGridSearch(SearchProtocol):
    """Search protocol for continuous input spaces using a fixed grid.

    This creates a dense grid of candidate points in the continuous input space.
    The acquisition function will then select the most promising points from this grid.

    Unlike SingleMutantSearch (for discrete sequences), this generates points
    uniformly across the continuous domain.
    """

    def __init__(self, x_min: float, x_max: float, n_points: int = 500):
        """Initialize the continuous grid search.

        Args:
            x_min: Minimum value of the input range
            x_max: Maximum value of the input range
            n_points: Number of grid points to create
        """
        self.x_min = x_min
        self.x_max = x_max
        self.n_points = n_points

    def __call__(self, task_state: TaskState) -> list[Candidate]:
        """Generate a grid of candidate points.

        Args:
            task_state: Current task state (not used, but required by interface)

        Returns:
            List of candidate points on the grid
        """
        x_grid = np.linspace(self.x_min, self.x_max, self.n_points)
        candidates = [Candidate(data=np.array([x]), modality=Modality.TABULAR) for x in x_grid]
        return candidates


print("✅ Custom ContinuousGridSearch created!")
print("🔍 This search protocol generates a grid of points in continuous space.")
print("   Unlike SingleMutantSearch (for discrete sequences), this works for continuous x values.")

### Step 9: Configure Search Strategy and Acquisition Function

Now we'll set up the search strategy and acquisition function:

In [ ]:
# Create search strategy with continuous grid
from alf_tools.optimizer.acquisition_functions.threshold_ucb import ThresholdUCB

# Create search strategy with continuous grid
search_fn = ProtocolSearch(
    protocol=ContinuousGridSearch(x_min=X_MIN, x_max=X_MAX, n_points=CANDIDATE_GRID_SIZE)
)

# Create acquisition function (ThresholdUCB balances exploration and exploitation)
acquisition_fn = ThresholdUCB(
    threshold=ACQ_THRESHOLD,
    beta=ACQ_BETA,
    exploration_weight=ACQ_EXPLORATION_WEIGHT,
    diversity_penalty=ACQ_DIVERSITY_PENALTY,
    diversity_length_scale=ACQ_DIVERSITY_LENGTH_SCALE,
)

# Create optimizer
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

print("✅ Optimizer configured!")
print("🔍 Search strategy:")
print("   - Type: Continuous grid search")
print(f"   - Grid size: {CANDIDATE_GRID_SIZE} candidate points")
print("   - Domain: [0, 2π]")
print("🎯 Acquisition strategy:")
print("   - Type: Greedy (selects highest predicted values)")
print("   - Uses GP predictions to select informative points")

### Step 10: Configure and Run the Design Task

Now we'll run the multi-round online design experiment. Each round:
1. **Ask**: Optimizer proposes candidates to evaluate
2. **Oracle Query**: Simulator evaluates selected points
3. **Tell**: GP is retrained with new data
4. **Evaluate**: Performance metrics are computed

In [ ]:
# Configure the design task
task = DesignTask(num_acq_rounds=NUM_ACQ_ROUNDS, acq_batch_size=ACQ_BATCH_SIZE)

print("✅ Design task configured!")
print("📋 Experiment parameters:")
print(f"   - Acquisition rounds: {NUM_ACQ_ROUNDS}")
print(f"   - Batch size per round: {ACQ_BATCH_SIZE}")
print(f"   - Total new points to acquire: {NUM_ACQ_ROUNDS * ACQ_BATCH_SIZE}")
print(f"   - Initial training points: {len(dataset.train_dataset)}")
print(f"   - Final training points: {len(dataset.train_dataset) + NUM_ACQ_ROUNDS * ACQ_BATCH_SIZE}")

### Step 11: Run the Active Learning Experiment

Let's run the complete online design experiment:

In [ ]:
# Set up loggers
terminal_logger = TerminalTaskStateLogger()
save_path = Path("results/gp_online_design/")
if save_path.exists():
    shutil.rmtree(save_path)
file_logger = FileTaskStateLogger(output_path=save_path)
loggers = [file_logger, terminal_logger]

# Set up the initial state
print("🚀 Setting up online design experiment...")
state = task.setup(dataset=dataset, surrogate=surrogate)

print("\n📊 Initial state:")
print(f"   - Training samples: {len(state.dataset.train_dataset)}")
print(f"   - Validation samples: {len(state.dataset.validation_dataset)}")

# Run the active learning experiment
print("\n🔄 Starting active learning loop...")
print("=" * 50)

task.run(state=state, task_state_loggers=loggers, optimizer=optimizer, oracle=oracle)

print("\n✅ Online design experiment completed!")
print(f"📊 Total simulator evaluations: {simulator.num_evaluations}")

### Step 12: Analyze the Results

Let's examine how the active learning process evolved over the rounds:

In [ ]:
# Load metrics from CSV
metrics = pd.read_csv("results/gp_online_design/metrics.csv")

# Print summary statistics
print("📊 Experiment Summary:")
print("=" * 50)
print(f"Total Rounds: {len(metrics)}")
print(f"Initial mean fitness: {metrics['acquired_candidates/round_mean'].iloc[1]:.4f}")
print(f"Final mean fitness: {metrics['acquired_candidates/round_mean'].iloc[-1]:.4f}")
print(f"Best fitness found: {metrics['acquired_candidates/round_max'].max():.4f}")
print(f"Total points acquired: {NUM_ACQ_ROUNDS * ACQ_BATCH_SIZE}")
print(f"Total simulator queries: {simulator.num_evaluations}")

### Step 13: Visualize Active Learning Progress

Let's visualize how the acquired candidates evolved over rounds:

In [ ]:
# Create visualization of acquisition progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Online Design Optimization Results", fontsize=16, fontweight="bold")

# Filter out the zeroth round (initial training round)
metrics_plot = metrics.iloc[1:]

# Left: Mean fitness of acquired batches
axes[0].plot(
    metrics_plot["round"],
    metrics_plot["acquired_candidates/round_mean"],
    marker="o",
    linewidth=2,
    markersize=8,
    color="#e74c3c",
)
axes[0].set_xlabel("Round", fontsize=12)
axes[0].set_ylabel("Mean Fitness", fontsize=12)
axes[0].set_title("Acquired Batch Mean Fitness", fontsize=13, fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Right: Max fitness of acquired batches
axes[1].plot(
    metrics_plot["round"],
    metrics_plot["acquired_candidates/round_max"],
    marker="s",
    linewidth=2,
    markersize=8,
    color="#2ecc71",
)
axes[1].set_xlabel("Round", fontsize=12)
axes[1].set_ylabel("Maximum Fitness", fontsize=12)
axes[1].set_title("Acquired Batch Maximum Fitness", fontsize=13, fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 The GP-guided active learning discovers high-fitness regions efficiently!")

### Step 14: Visualize Final GP Predictions

Let's see how well the GP learned the sinusoidal function after all acquisition rounds:

In [ ]:
# Get all training data (initial + acquired)
all_train_x = np.array([c.data[0] for c in state.dataset.train_dataset.candidates])
all_train_y = state.dataset.train_dataset.labels

# Separate initial vs acquired points for visualization
n_initial = len(initial_data)
initial_x = all_train_x[:n_initial]
initial_y = all_train_y[:n_initial]
acquired_x = all_train_x[n_initial:]
acquired_y = all_train_y[n_initial:]

# Make predictions on dense grid
x_test = np.linspace(X_MIN, X_MAX, N_TEST_POINTS)
test_candidates = [Candidate(data=np.array([xi]), modality=Modality.TABULAR) for xi in x_test]
predictions = gp_model.predict(test_candidates)
pred_means = predictions.means
pred_stds = np.sqrt(predictions.variances)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Final Gaussian Process After Active Learning", fontsize=16, fontweight="bold")

# Left: GP predictions with all data
axes[0].scatter(
    initial_x, initial_y, alpha=0.7, label="Initial training data", color="blue", s=80, zorder=3
)
axes[0].scatter(
    acquired_x,
    acquired_y,
    alpha=0.7,
    label="Acquired data",
    color="red",
    s=80,
    marker="^",
    zorder=3,
)
axes[0].plot(x_test, pred_means, "purple", label="GP mean", linewidth=2.5)
axes[0].fill_between(
    x_test,
    pred_means - 2 * pred_stds,
    pred_means + 2 * pred_stds,
    alpha=0.3,
    color="purple",
    label="95% confidence",
)
axes[0].plot(x_test, sample_function(x_test), "g--", label="True function", linewidth=2, alpha=0.7)
axes[0].plot(
    x_test, [ACQ_THRESHOLD] * len(x_test), "b--", label="Threshold", linewidth=1, alpha=0.7
)
axes[0].set_xlabel("x", fontsize=12)
axes[0].set_ylabel("y", fontsize=12)
axes[0].set_title("GP Predictions with Acquired Data", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: Prediction uncertainty
axes[1].plot(x_test, pred_stds, "purple", linewidth=2.5, label="Prediction uncertainty")
axes[1].scatter(
    initial_x,
    np.zeros_like(initial_x),
    alpha=0.5,
    color="blue",
    s=80,
    label="Initial data",
    zorder=3,
)
axes[1].scatter(
    acquired_x,
    np.zeros_like(acquired_x),
    alpha=0.5,
    color="red",
    s=80,
    marker="^",
    label="Acquired data",
    zorder=3,
)
axes[1].set_xlabel("x", fontsize=12)
axes[1].set_ylabel("Prediction Std Dev", fontsize=12)
axes[1].set_title("Prediction Uncertainty After Active Learning", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Key observations:")
print("   - Active learning acquired points strategically")
print("   - GP now has better coverage across the input space")
print("   - Uncertainty is reduced in regions with more data")
print("   - The acquisition strategy used GP predictions to guide exploration")

### Step 15: Visualize Acquisition History

Let's see how points were acquired over time:

In [ ]:
# Visualize where points were acquired in each round
plt.figure(figsize=(12, 6))

# Plot true function
x_dense = np.linspace(X_MIN, X_MAX, N_TEST_POINTS)
plt.plot(x_dense, sample_function(x_dense), "k--", label="True function", linewidth=2, alpha=0.5)

# Plot initial points
plt.scatter(
    initial_x,
    initial_y,
    alpha=0.8,
    label="Initial data",
    color="blue",
    s=100,
    zorder=3,
    edgecolors="black",
    linewidths=1.5,
)

# Plot acquired points by round with different colors
colors = plt.cm.Reds(np.linspace(0.4, 1, NUM_ACQ_ROUNDS))
for i in range(NUM_ACQ_ROUNDS):
    start_idx = n_initial + i * ACQ_BATCH_SIZE
    end_idx = start_idx + ACQ_BATCH_SIZE
    round_x = all_train_x[start_idx:end_idx]
    round_y = all_train_y[start_idx:end_idx]
    plt.scatter(
        round_x,
        round_y,
        alpha=0.8,
        label=f"Round {i + 1}",
        color=colors[i],
        s=100,
        zorder=3,
        edgecolors="black",
        linewidths=1.5,
    )

plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.title("Point Acquisition Over Time", fontsize=14, fontweight="bold")
plt.legend(fontsize=10, loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📍 The visualization shows when and where each point was acquired during active learning.")

### Step 16: Compare with Random Sampling (Optional)

To demonstrate the value of active learning, let's compare with random sampling:

In [ ]:
# Simulate random sampling baseline
np.random.seed(123)  # Different seed for comparison

# Random sampling: same total number of points as active learning
n_total = len(all_train_x)
x_random = np.random.uniform(X_MIN, X_MAX, n_total)
random_candidates = [Candidate(data=np.array([x]), modality=Modality.TABULAR) for x in x_random]

# Create new simulator for fair comparison
simulator_random = SinusoidalSimulator(noise_std=SIMULATOR_NOISE_STD, seed=RANDOM_COMPARISON_SEED)
predictions_random = simulator_random.predict(random_candidates)
y_random = predictions_random.means

# Train a new GP on random data
gp_random = GPModelTrainer(
    name="gp_random",
    model_config=model_config,
    train_config=train_config,
    featurizer_config=featurizer_config,
)

random_labeled_data = LabeledCandidates(random_candidates, y_random)
gp_random.train(random_labeled_data)

# Make predictions with random GP
pred_random = gp_random.predict(test_candidates)
pred_means_random = pred_random.means

# Compare predictions
y_true_test = sample_function(x_test)
mse_active = np.mean((pred_means - y_true_test) ** 2)
mse_random = np.mean((pred_means_random - y_true_test) ** 2)

print("\n🔬 Active Learning vs Random Sampling:")
print("=" * 50)
print(f"Number of points: {n_total} (same for both)")
print("\nMSE on true function:")
print(f"   - Active learning: {mse_active:.4f}")
print(f"   - Random sampling: {mse_random:.4f}")
print(f"   - Improvement: {((mse_random - mse_active) / mse_random * 100):.1f}%")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Active Learning vs Random Sampling", fontsize=16, fontweight="bold")

# Left: Active learning
axes[0].scatter(all_train_x, all_train_y, alpha=0.5, color="red", s=50, label="Training data")
axes[0].plot(x_test, pred_means, "purple", linewidth=2, label="GP prediction")
axes[0].plot(x_test, y_true_test, "g--", linewidth=2, label="True function")
axes[0].set_xlabel("x", fontsize=12)
axes[0].set_ylabel("y", fontsize=12)
axes[0].set_title(f"Active Learning (MSE: {mse_active:.4f})", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Random sampling
axes[1].scatter(x_random, y_random, alpha=0.5, color="blue", s=50, label="Training data")
axes[1].plot(x_test, pred_means_random, "orange", linewidth=2, label="GP prediction")
axes[1].plot(x_test, y_true_test, "g--", linewidth=2, label="True function")
axes[1].set_xlabel("x", fontsize=12)
axes[1].set_ylabel("y", fontsize=12)
axes[1].set_title(f"Random Sampling (MSE: {mse_random:.4f})", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Step 17: Clean Up

Optionally, clean up the results directory:

In [ ]:
# Optionally, clean up the results directory
if save_path.exists():
    shutil.rmtree(save_path)

print("✅ Results directory cleaned up!")

## Conclusion

This tutorial has demonstrated how to use **Gaussian Processes for online design optimization** with the ALF framework. We've learned:

- **How to create an online simulator** that acts as an oracle for on-demand evaluation
- **How to create custom search protocols** for continuous spaces (unlike `SingleMutantSearch` for discrete sequences)
- **How to set up a design task** with active learning for iterative data acquisition
- **How to use GP uncertainty** to guide exploration and exploitation
- **How to configure the ask-tell cycle** with optimizer, acquisition function, and search strategy
- **How to visualize active learning progress** and understand where the GP focuses queries
- **How to compare with random sampling** to demonstrate active learning efficiency

### Key Advantages of GP-based Active Learning:

1. **Uncertainty-Guided Exploration**: GP uncertainty naturally guides where to sample next
2. **Sample Efficiency**: Achieves better performance with fewer evaluations than random sampling
3. **Online Learning**: Adapts as new data becomes available
4. **Principled Decision Making**: Bayesian framework provides theoretically grounded acquisition
5. **Flexible**: Works with any expensive evaluation function (experiments, simulations, etc.)

### Real-World Applications:

This approach is valuable for:
- **Protein engineering**: Expensive lab experiments or MD simulations
- **Drug discovery**: High-throughput screening campaigns
- **Materials design**: Experimental synthesis and testing
- **Hyperparameter optimization**: Expensive model training
- **Scientific experiments**: Any domain with costly evaluations

### Next Steps:

- Try different acquisition functions (UCB, Thompson sampling, Expected Improvement)
- Experiment with different GP kernels for various function types
- Scale to higher dimensional problems
- Use batch acquisition strategies for parallel evaluation
- Apply to your own domain-specific problems!

**Happy optimizing!** 🚀🔬✨